# Relay link table: radio maps from every drivable cell

Generates a predicted 5.8 GHz radio map (RadioLunaDiff: K2-UNet → PMNet → diffusion residual,
the same pipeline as `rm_generate.py`) with the transmitter at **every drivable cell** of a site,
and keeps the RSS at every drivable cell. The result is a link table for the relay rover:

`rss[i, j]` = predicted RSS [dBm, rounded to 1 dB, int8] at drivable cell `rx_rc[j]`
with the transmitter at candidate cell `tx_rc[i]`.

**Setup** (once per site):
1. On the laptop: `python rm_colab_bundle.py --site fountain` → `rm_colab_fountain.zip`.
2. Put that zip and lunar-mesh-env's `RadioLunaDiff/pretrained_models_network.zip` (the weights,
   632 MB, same for every site) in one Google Drive folder, `MyDrive/rm_linktab` by default.
3. Runtime → Change runtime type → a GPU (T4 is enough). Run all cells.

Output goes to `MyDrive/rm_linktab/out/<site>/<site>_linktab_<model>.npz`. The run is saved in
shards as it goes, so after a disconnect just run all cells again and it carries on.

In [ ]:
SITE = 'fountain'
DRIVE_DIR = '/content/drive/MyDrive/rm_linktab'   # holds rm_colab_<site>.zip + pretrained_models_network.zip
MODELS = ['nn_pmnet', 'nn']   # nn_pmnet: PMNet stage only (deterministic); nn: + diffusion residual (the dashboard's "nn")
TX_SPACING_M = 0.0            # 0 = every drivable cell is a transmitter candidate; e.g. 2.0 = candidates >= 2 m apart
BATCH = 32                    # maps per GPU batch (halved automatically on out-of-memory)
SHARD = 512                   # transmitter cells per saved shard
SEED = 0                      # diffusion noise seed
FREQ = '5.8'
LIMIT = None                  # only the first N candidates (quick test); None = all

In [ ]:
import json
import math
import os
import subprocess
import sys
import time
import zipfile

import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/work'
    # Colab's torchao doesn't match its diffusers ("cannot import name 'FqnToConfig'") and nothing
    # here needs it. diffusers looks for torchao once, at import: if this cell already failed after
    # diffusers was imported, do Runtime -> Restart session, then run all again.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
else:                                            # local test run: zips in RM_DIR, work dir next to them
    DRIVE_DIR = os.environ.get('RM_DIR', DRIVE_DIR)
    WORK = os.path.join(DRIVE_DIR, 'work')
try:
    import diffusers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers'], check=True)

import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, torch.cuda.get_device_name(0) if dev == 'cuda' else '(no GPU: this will be slow)')

In [ ]:
bundle = os.path.join(DRIVE_DIR, f'rm_colab_{SITE}.zip')
weights = os.path.join(DRIVE_DIR, 'pretrained_models_network.zip')
for z in (bundle, weights):
    assert os.path.exists(z), f'missing {z}'
os.makedirs(WORK, exist_ok=True)
MODEL_DIR = os.path.join(WORK, 'pretrained_models_network')
if not os.path.exists(os.path.join(MODEL_DIR, 'pmnet', 'best_pm_model.pt')):
    print('unpacking weights...')
    zipfile.ZipFile(weights).extractall(WORK)
zipfile.ZipFile(bundle).extractall(WORK)
sys.path.insert(0, os.path.join(WORK, 'code'))

OUT = os.path.join(DRIVE_DIR, 'out', SITE)
SHARD_DIR = os.path.join(OUT, 'shards')
os.makedirs(SHARD_DIR, exist_ok=True)

## Site and candidate cells
Map cells: row 0 = north, col 0 = west, 1 m (see `site_map.py`). Receivers are always every
drivable cell; transmitter candidates are the same cells, optionally thinned to `TX_SPACING_M`.

In [ ]:
T = os.path.join(WORK, 'terrain')
meta = json.load(open(os.path.join(T, f'{SITE}_meta.json')))
cell_m = float(meta.get('cell_m', 1.0))
hm = np.load(os.path.join(T, f'{SITE}_terrain.npy'))
drv = np.load(os.path.join(T, f'{SITE}_drivable.npy')).astype(bool)
size = hm.shape[0]
n = size - 1

rx_rc = np.argwhere(drv).astype(np.int16)        # row-major, the same order as idx[drv]
idx = np.full(drv.shape, -1, np.int32)
idx[drv] = np.arange(len(rx_rc))


def thin(cells, spacing_m):
    """Greedy thinning: keep a cell unless one already kept is closer than spacing_m."""
    if spacing_m <= 0:
        return cells
    rad = spacing_m / cell_m
    R = int(math.ceil(rad))
    taken = np.zeros(drv.shape, bool)
    keep = []
    for r, c in cells.astype(int):
        r0, c0 = max(r - R, 0), max(c - R, 0)
        near = np.argwhere(taken[r0:r + R + 1, c0:c + R + 1])
        if len(near) and (((near[:, 0] + r0 - r) ** 2 + (near[:, 1] + c0 - c) ** 2) < rad * rad).any():
            continue
        taken[r, c] = True
        keep.append((r, c))
    return np.array(keep, np.int16).reshape(-1, 2)


tx_rc = thin(rx_rc, TX_SPACING_M)
if LIMIT:
    tx_rc = tx_rc[:LIMIT]
tx_idx = np.full(drv.shape, -1, np.int32)
tx_idx[tx_rc[:, 0], tx_rc[:, 1]] = np.arange(len(tx_rc))
TAG = f'sp{TX_SPACING_M:g}_seed{SEED}' + (f'_lim{LIMIT}' if LIMIT else '')
print(f'{SITE}: {size}x{size} cells of {cell_m:g} m, {len(rx_rc)} drivable (receivers), '
      f'{len(tx_rc)} transmitter candidates')

## Load the model

In [ ]:
from lunar_mesh_env.radio_model_nn import RadioMapModelNN

# the model puts the transmitter at pixel (x / width * (size - 1)): width = size - 1 => pixel = (col, row)
model = RadioMapModelNN({'k2_model': os.path.join(MODEL_DIR, 'k2unet', 'best_k2_model.pth'),
                         'pmnet_model': os.path.join(MODEL_DIR, 'pmnet', 'best_pm_model.pt'),
                         'diffusion_model': os.path.join(MODEL_DIR, 'diffusion')},
                        hm, float(n), float(n), device=dev)


@torch.no_grad()
def run_batch(cells_rc, want_nn):
    """Radio maps [dBm] for transmitters at cells (row, col): {model: tensor (B, size, size)}.
    Same steps as RadioMapModelNN._run_batch_inference, keeping the PMNet stage as well."""
    m, p = model, model.processor
    B = len(cells_rc)
    tx = torch.stack([p.create_tx_map((float(c), float(r)), n, n, m.device) for r, c in cells_rc])
    x = torch.stack([m.static_hm_tensor.expand(B, -1, -1), tx,
                     p.create_freq_map(FREQ, m.device).expand(B, -1, -1),
                     m.static_fhm_tensor.expand(B, -1, -1)], 1)
    k2 = torch.sigmoid(m.k2_model(x.clone()))
    pm_n = m.normalize_transform_single(m.pmnet(torch.cat([x.clone(), k2], 1)))
    out = {'nn_pmnet': pm_n}
    if want_nn:
        cond, k2_n = m.normalize_transform(x.clone()), m.normalize_transform_single(k2)
        g = torch.randn((B, 1, size, size), device=m.device)
        for t in m.noise_scheduler.timesteps:
            noise = m.model(torch.cat([g, cond, pm_n, k2_n], 1), t, return_dict=False)[0]
            g = m.noise_scheduler.step(noise, t, g, return_dict=False)[0]
        out['nn'] = pm_n + g
    return {k: ((v.clamp(-1, 1) + 1.0) / 2.0 * 210.0 - 200.0)[:, 0] for k, v in out.items()}

## Check against the site's own maps
The site's BS cell (`<site>_tx.npy`) has maps made on the laptop. `nn_pmnet` is deterministic,
so it should match to a fraction of a dB (GPU vs CPU rounding); `nn` has its own noise, so expect
a few dB of scatter but the same overall picture.

In [ ]:
import matplotlib.pyplot as plt

tx_file = os.path.join(T, f'{SITE}_tx.npy')
if os.path.exists(tx_file) and np.load(tx_file).any():
    bs_rc = tuple(int(v) for v in np.argwhere(np.load(tx_file))[0])
    torch.manual_seed(SEED)
    chk = {k: v[0].float().cpu().numpy() for k, v in run_batch([bs_rc], 'nn' in MODELS).items()}
    fig, axs = plt.subplots(1, len(chk), figsize=(5 * len(chk), 4.5), squeeze=False)
    for ax, (k, arr) in zip(axs[0], chk.items()):
        ref_p = os.path.join(T, f'{SITE}_rm58_{k}.npy')
        if os.path.exists(ref_p):
            d = np.abs(arr - np.load(ref_p))[drv]
            print(f'{k}: vs laptop map, on drivable cells: mean |diff| {d.mean():.2f} dB, max {d.max():.2f} dB')
        im = ax.imshow(arr, cmap='viridis', vmin=-110, vmax=-20)
        ax.plot(bs_rc[1], bs_rc[0], 'r^')
        ax.set_title(f'{k}, tx at BS {bs_rc}')
        plt.colorbar(im, ax=ax, label='dBm')
    plt.show()
else:
    bs_rc = None
    print('no tx.npy in the bundle: skipping the check')

## Generate (resumable)
Each shard of `SHARD` transmitters is saved to Drive as soon as it's done (float16 RSS at the
drivable cells); finished shards are skipped on a re-run.

In [ ]:
def shard_path(model_name, k):
    return os.path.join(SHARD_DIR, f'{model_name}_{TAG}_{k:04d}.npy')


n_sh = math.ceil(len(tx_rc) / SHARD)
rx_t = torch.as_tensor(rx_rc.astype(np.int64), device=dev)
todo = [k for k in range(n_sh) if not all(os.path.exists(shard_path(m, k)) for m in MODELS)]
print(f'{n_sh} shards, {len(todo)} to do')
t_start, done_maps = time.time(), 0
for k in todo:
    torch.manual_seed(SEED * 100003 + k)
    cells = tx_rc[k * SHARD:(k + 1) * SHARD]
    bufs = {m: np.empty((len(cells), len(rx_rc)), np.float16) for m in MODELS}
    i = 0
    while i < len(cells):
        chunk = cells[i:i + BATCH]
        try:
            out = run_batch(chunk, 'nn' in MODELS)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            BATCH = max(1, BATCH // 2)
            print('out of memory: BATCH ->', BATCH)
            continue
        for m in MODELS:
            bufs[m][i:i + len(chunk)] = out[m][:, rx_t[:, 0], rx_t[:, 1]].half().cpu().numpy()
        i += len(chunk)
    for m in MODELS:
        tmp = shard_path(m, k) + '.tmp.npy'
        np.save(tmp, bufs[m])
        os.replace(tmp, shard_path(m, k))
    done_maps += len(cells)
    rate = done_maps / (time.time() - t_start)
    left = sum(len(tx_rc[j * SHARD:(j + 1) * SHARD]) for j in todo[todo.index(k) + 1:])
    print(f'shard {k + 1}/{n_sh}: {rate:.1f} maps/s, ~{left / rate / 60:.1f} min left', flush=True)

## Assemble the link tables

In [ ]:
for m in MODELS:
    rss = np.concatenate([np.load(shard_path(m, k)) for k in range(n_sh)])
    q = np.clip(np.rint(rss.astype(np.float32)), -128, 127).astype(np.int8)
    info = {'site': SITE, 'model': m, 'freq_ghz': FREQ, 'seed': SEED, 'tx_spacing_m': TX_SPACING_M,
            'cell_m': cell_m, 'size': size, 'created': time.strftime('%Y-%m-%d %H:%M:%S'),
            'layout': 'rss[i, j] = predicted RSS [dBm, 1 dB steps] at rx_rc[j] with the transmitter '
                      'at tx_rc[i]; cells are (row, col), row 0 = north, col 0 = west; '
                      'idx / tx_idx map a cell to its rx / tx index (-1 = none)'}
    path = os.path.join(OUT, f'{SITE}_linktab_{m}{"" if TAG == "sp0_seed0" else "_" + TAG}.npz')
    np.savez(path, rss=q, tx_rc=tx_rc, rx_rc=rx_rc, idx=idx, tx_idx=tx_idx, meta=json.dumps(info))
    print(f'{path}: {q.shape[0]} x {q.shape[1]}, {os.path.getsize(path) / 1e6:.0f} MB')

## Look at the result
**Reciprocity**: a radio link loses the same in both directions, but the model predicts each
direction separately, so `rss[a→b]` and `rss[b→a]` differ somewhat.
**Relay example**: with the BS at its cell and the explorer at a far, badly covered spot, the
relay's best cell maximises the weaker of its two links, `min(BS→relay, relay→explorer)`.

In [ ]:
m = MODELS[-1]
L = np.load(os.path.join(OUT, f'{SITE}_linktab_{m}{"" if TAG == "sp0_seed0" else "_" + TAG}.npz'))
rss, t_rc, r_idx = L['rss'].astype(np.float32), L['tx_rc'], L['idx']

# reciprocity, over transmitter pairs that are also receivers
rx_of_tx = r_idx[t_rc[:, 0], t_rc[:, 1]]
rng = np.random.default_rng(0)
a, b = rng.integers(0, len(t_rc), (2, 200000))
d = np.abs(rss[a, rx_of_tx[b]] - rss[b, rx_of_tx[a]])
print(f'{m}: |rss(a->b) - rss(b->a)|: mean {d.mean():.1f} dB, 90% {np.percentile(d, 90):.1f} dB')

if bs_rc is not None:
    # nearest drivable cell stands in for the BS if the BS itself isn't on a path
    j_bs = int(np.argmin(((rx_rc - np.array(bs_rc)) ** 2).sum(1)))
    i_bs = int(np.argmin(((t_rc - np.array(bs_rc)) ** 2).sum(1)))
    # BS -> relay, averaged over both directions
    to_bs = 0.5 * (rss[:, j_bs] + rss[i_bs, rx_of_tx])
    # explorer: the cell with the weakest direct link that is still > 40 m from the BS
    far = np.hypot(*(rx_rc - np.array(bs_rc)).T) * cell_m > 40
    direct = rss[i_bs]
    j_a = int(np.where(far)[0][np.argmin(direct[far])])
    i_a = int(np.argmin(((t_rc - rx_rc[j_a]) ** 2).sum(1)))
    to_a = 0.5 * (rss[:, j_a] + rss[i_a, rx_of_tx])
    score = np.minimum(to_bs, to_a)
    best = int(np.argmax(score))
    print(f'explorer at {tuple(rx_rc[j_a])}: direct BS link {direct[j_a]:.0f} dBm; best relay at '
          f'{tuple(t_rc[best])}: BS {to_bs[best]:.0f} / explorer {to_a[best]:.0f} dBm')

    plt.figure(figsize=(7, 6.5))
    plt.imshow(~drv, cmap='gray', vmin=-1, vmax=1.5)
    sc = plt.scatter(t_rc[:, 1], t_rc[:, 0], c=score, s=2, cmap='viridis', vmin=-100, vmax=-40)
    plt.colorbar(sc, label='min(BS link, explorer link) [dBm]')
    plt.plot(bs_rc[1], bs_rc[0], 'r^', ms=10, label='BS')
    plt.plot(rx_rc[j_a, 1], rx_rc[j_a, 0], 'ms', ms=8, label='explorer')
    plt.plot(t_rc[best, 1], t_rc[best, 0], 'w*', ms=14, mec='k', label='best relay')
    plt.legend(loc='lower right')
    plt.title(f'{SITE}: relay score ({m})')
    plt.show()